In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import statsmodels.api as sm

In [2]:
mRNA = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_gene_protein.xlsx')
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
mRNA = mRNA.loc[:, ~mRNA.columns.str.contains('Norm')]
mRNA = mRNA.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T
#trasponemos para que cada fila sea un paciente y cada columna sea una variable , incluimos el simbolo de hgnc como nombre de columnas  y eliminamos variables que son string
mRNA = mRNA[['ACTA1', 'TNNT1', 'CYP2J2', 'HSPB6', 'MYL2', 'LTBP2', 'XPR1', 'SORT1',
       'PACS1', 'MFGE8', 'FSTL3', 'FGF12']]
lncRNA = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_LncRNA_identificados.xlsx')
lncRNA = lncRNA.loc[:, ~lncRNA.columns.str.contains('Norm')]
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
lncRNA = lncRNA.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T
lncRNA = lncRNA[['MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208', 'LINC00702',
       'TNRC6C-AS1', 'H19']]

### Correlations

In [115]:
cor_matrix = pd.DataFrame(index=df_protein.columns, columns=df_noCoding.columns)
for gene in df_protein.columns:
    for lnc in df_noCoding.columns:
        rho, _ = spearmanr(df_protein[gene], df_noCoding[lnc])
        cor_matrix.loc[gene, lnc] = rho
corr_matrix = cor_matrix.reset_index().rename(columns = {'hgnc_symbol' : 'lncRNA'})
corr_matrix['Grupo'] = [1,1,1,1,1,2,2,2,2,2,2,3]

In [116]:
df_protein2 = df_protein.copy().reset_index()
df_protein2['Grupo'] = np.where(df_protein2['index'].str.contains('A'), 'Tratamiento', 
            np.where(df_protein2['index'].str.contains('C'), 'Control', 'Otro'))
df_protein2 = df_protein2.set_index('index')
df_protein2 = pd.get_dummies(df_protein2, columns=['Grupo'], drop_first=True)
df_protein2['Grupo_Tratamiento'] = df_protein2['Grupo_Tratamiento'].astype(int)
grupo_control = df_protein2[df_protein2['Grupo_Tratamiento'] == 0].drop(columns='Grupo_Tratamiento')
grupo_tratamiento = df_protein2[df_protein2['Grupo_Tratamiento'] == 1].drop(columns='Grupo_Tratamiento')
mean_control = grupo_control.mean()
mean_tratamiento = grupo_tratamiento.mean()
logFC = np.log2(mean_tratamiento / mean_control)
logFC_df = pd.DataFrame({'log2FC': logFC})
logFC_df = logFC_df.reset_index().rename(columns = {'index' : 'lncRNA'})
corr_matrix = pd.merge(corr_matrix,logFC_df, on = 'lncRNA', how = 'inner')
corr_matrix['Up/Down'] =  np.where(corr_matrix['log2FC'] > 0, 'Up', 'Down')
corr_matrix = corr_matrix[['lncRNA','Grupo','log2FC','Up/Down','MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208',
       'LINC00702', 'TNRC6C-AS1', 'H19']]

In [117]:
corr_matrix.to_csv('Tabla_corr_variablesRepresentativas.csv',index = False)

Lo hacemos con todas las variables, las 78, no solo las representativas

In [118]:
df_protRelTotal = pd.read_csv('df_proteines_totalRelevant.csv',index_col = 0)
df_protcluster = pd.read_csv('vars_cluster.csv')

In [119]:
cor_matrix = pd.DataFrame(index=df_protRelTotal.columns, columns=df_noCoding.columns)
for gene in df_protRelTotal.columns:
    for lnc in df_noCoding.columns:
        rho, _ = spearmanr(df_protRelTotal[gene], df_noCoding[lnc])
        cor_matrix.loc[gene, lnc] = rho
corr_matrix = cor_matrix.reset_index().rename(columns = {'hgnc_symbol' : 'lncRNA'})
corr_matrix = corr_matrix.rename(columns = {'index' : 'variable'})
corr_matrix = corr_matrix.merge(df_protcluster, on = 'variable', how = 'inner').sort_values('grupo').reset_index(drop = True).rename(columns = {'variable' : 'lncRNA'})

In [120]:
df_protein2 = df_protRelTotal.copy().reset_index()
df_protein2['Grupo'] = np.where(df_protein2['index'].str.contains('A'), 'Tratamiento', 
            np.where(df_protein2['index'].str.contains('C'), 'Control', 'Otro'))
df_protein2 = df_protein2.set_index('index')
df_protein2 = pd.get_dummies(df_protein2, columns=['Grupo'], drop_first=True)
df_protein2['Grupo_Tratamiento'] = df_protein2['Grupo_Tratamiento'].astype(int)
grupo_control = df_protein2[df_protein2['Grupo_Tratamiento'] == 0].drop(columns='Grupo_Tratamiento')
grupo_tratamiento = df_protein2[df_protein2['Grupo_Tratamiento'] == 1].drop(columns='Grupo_Tratamiento')
mean_control = grupo_control.mean()
mean_tratamiento = grupo_tratamiento.mean()
logFC = np.log2(mean_tratamiento / mean_control)
logFC_df = pd.DataFrame({'log2FC': logFC})
logFC_df = logFC_df.reset_index().rename(columns = {'index' : 'lncRNA'})
corr_matrix = pd.merge(corr_matrix,logFC_df, on = 'lncRNA', how = 'inner')
corr_matrix['Up/Down'] =  np.where(corr_matrix['log2FC'] > 0, 'Up', 'Down')
corr_matrix = corr_matrix[['lncRNA','grupo','log2FC','Up/Down','MBNL1-AS1', 'LINC01278', 'MIR1-1HG-AS1', 'LINC02208',
       'LINC00702', 'TNRC6C-AS1', 'H19']].sort_values('grupo')


In [121]:
corr_matrix.to_csv('Tabla_corr_Todas_Variables.csv',index = False)

### ML

In [66]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics  import r2_score

In [ ]:
def scl(df):
    scaler = StandardScaler()
    df_scl = pd.DataFrame(scaler.fit_transform(df), columns = df.columns, index=df.index)
    return df_scl
def pca90(df, prefix):
    scaler = StandardScaler()
    df_scl = scaler.fit_transform(df)
    pca = PCA(n_components=0.9)

    df_scl_comp = pca.fit_transform(df_scl)
    col_names = [f"{prefix}_comp{i+1}" for i in range(df_scl_comp.shape[1])]
    return pd.DataFrame(df_scl_comp, index = df.index, columns = col_names)
def discretizar_variables(df, q = 3):
    """
    Discretiza las variables continuas usando una cantidad definida de intervalos.
    """
    df_discretized = df.copy()
    for col in df.columns:
        df_discretized[col] = pd.qcut(df[col], q=q, labels=False, duplicates='drop')
    data_encoded = pd.DataFrame()
    for col in df_discretized.columns:
        dummies = pd.get_dummies(df_discretized[col], prefix=col, prefix_sep='_')
        data_encoded = pd.concat([data_encoded, dummies], axis=1)
    data_encoded = data_encoded.astype(int)

    return data_encoded
def model_reg (X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_train_pred = lr.predict(X_train)
    y_test_pred = lr.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test

def model_rf(X,y,n_estimators = 100,max_depth = 2):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, oob_score=True, random_state=42)
    rf.fit(X_train, y_train)
    y_train_pred = rf.predict(X_train)
    y_test_pred = rf.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test

def regression_expl(df_x, df_y, model):
    r2_set = {}
    for col_y in df_y.columns:
        y = df_y[col_y]
        X = df_x
        r2_train, r2_test = model(X, y)
        r2_set.update({col_y : {'r2_train':r2_train , 'r2_test':r2_test,'r2_diff' : abs(r2_train - r2_test)}})
    return pd.DataFrame(r2_set)


Con PCA previo

In [75]:
lncRNA_pca = pca90(lncRNA, 'lncRNA')
mRNA_scl = scl(mRNA)
regression_expl(df_x = lncRNA_pca, df_y = mRNA_scl, model = model_reg)


,ACTA1,TNNT1,CYP2J2,HSPB6,MYL2,LTBP2,XPR1,SORT1,PACS1,MFGE8,FSTL3,FGF12
r2_train,0.744996,0.635573,0.813716,0.619241,0.640089,0.737236,0.900160,0.911406,0.794515,0.862705,0.871632,0.342992
r2_test,0.747298,0.483641,0.607433,0.755323,0.318688,0.537884,0.883175,0.908652,0.757018,0.709893,0.813080,0.485579
r2_diff,0.002302,0.151932,0.206283,0.136083,0.321401,0.199352,0.016986,0.002754,0.037497,0.152812,0.058552,0.142587


In [76]:
regression_expl(df_x = lncRNA_pca, df_y = mRNA_scl, model = model_rf)

,ACTA1,TNNT1,CYP2J2,HSPB6,MYL2,LTBP2,XPR1,SORT1,PACS1,MFGE8,FSTL3,FGF12
r2_train,0.789414,0.809474,0.856293,0.812350,0.816374,0.856381,0.901272,0.882394,0.867400,0.922413,0.841612,0.564137
r2_test,0.713966,0.416474,0.633371,0.484198,0.309924,0.381936,0.659457,0.575635,0.737745,0.225165,0.781027,0.206109
r2_diff,0.075448,0.393000,0.222921,0.328152,0.506450,0.474446,0.241815,0.306759,0.129656,0.697248,0.060585,0.358027


### Variables Instrumentales

In [164]:
def scl(df):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    df_scl = pd.DataFrame(scaler.fit_transform(df), columns = df.columns, index=df.index)
    return df_scl
def get_vars_significativas(model):
    pvals = model.pvalues
    significant_vars_i = pvals[pvals < 0.1].index.tolist()
    return significant_vars_i
def var_endogena(y,X,Z):
    X_significativas_list = []
    vars_endogenas = []
    modelos = []
    Z_const_new_list = []
    for x in X.columns:
        Z_const = sm.add_constant(Z)
        model2 = sm.OLS(X[x], Z_const).fit()
        sig_vars  = get_vars_significativas(model2)
        if not sig_vars or all([var == 'const' for var in sig_vars]):
            continue
        sig_vars = [var for var in sig_vars if var != 'const']
        if not sig_vars:
            continue
        Z_const_new = Z_const[sig_vars]
        model2 = sm.OLS(X[x], Z_const_new).fit()
        X_hat = model2.fittedvalues
        X_new = pd.concat([X_hat, X.drop(x, axis = 1)], axis = 1).rename(columns = {0 : f'{x}_hat'})
        X_new_const = sm.add_constant(X_new)
        model3 = sm.OLS(y, X_new_const).fit()
        X_significativas = X_new_const[get_vars_significativas(model3)]
        if f'{x}_hat' in X_significativas:
            #print(x, 'es variable endógena')
            X_significativas_sin_endog_list = [i for i in X_significativas.columns if i != f'{x}_hat' and i != "const"]
            X_significativas_list.append(X_significativas_sin_endog_list)
            vars_endogenas.append(x)
            modelos.append(model3)
            Z_const_new_list.append(Z_const_new)
        #else:
            #print(x, 'NO es variable endógena')
    return X_significativas_list, modelos, vars_endogenas,Z_const_new_list
def instrumentos_validos(y,X,Z, printable  = True):
    from scipy.stats import chi2
    instr_validos = []
    X_significativas_list, modelos, vars_endogenas,Z_const_new_list = var_endogena(y = y,X = X,Z = Z)
    for var_endog in range(len(vars_endogenas)):
        X_significativas = X_significativas_list[var_endog]
        model3 = modelos[var_endog]
        residuos3 = model3.resid
        Z_const_new = Z_const_new_list[var_endog]
        sargan_test  = sm.OLS(residuos3, Z_const_new).fit()
        sargan_test.summary()
        r2_sargan = sargan_test.rsquared
        n = len(y)
        sargan_stat = n * r2_sargan
        df_sargan = Z_const_new.shape[1] - 1 

        p_value = 1 - chi2.cdf(sargan_stat, df_sargan)
        if p_value > 0.1:
            if printable: 
                print(vars_endogenas[var_endog], 'causa Y a través de instrumentos válidos y exógenos:', list(Z_const_new.columns))
            X_significativas.append({vars_endogenas[var_endog]: list(Z_const_new.columns)})
            instr_validos.append(X_significativas)
        else:
            if printable:
                print(vars_endogenas[var_endog],'no tiene instrumentos válidos o exógenos')
    return instr_validos

def instrumentos_validos_ZGrande (y,X,Z,q_interactions):

    import random
    columnas =list(Z.columns)
    Z_tree_dict = {}
    iteracion = 0
    for _ in range(q_interactions):
        seleccionadas = random.sample(columnas, 5)
        Z_seleccionadas = Z[seleccionadas]
        Z_tree = instrumentos_validos(y = y,X = X,Z = Z_seleccionadas, printable  = False)
        if Z_tree:
            Z_tree_dict.update({iteracion : Z_tree})
            iteracion += 1
    return Z_tree_dict#pd.DataFrame(Z_tree_dict).T

def estimacion_IV(y,X,Z,zTree):
    modelosPorZtree = {}
    ztree_iter = 0
    for ztree_i in zTree:
        modelos = {}
        df = pd.DataFrame()
        for var in ztree_i:
            if type(var) == str:
                df[var] = X[var]
            else:
                endog = X[list(var.keys())[0]]
                instrs = Z[list(var.values())[0]]
                instrs = sm.add_constant(instrs)
                model_endog = sm.OLS(endog, instrs).fit()
                modelos.update({'instrumentModel':model_endog})
                X_hat = model_endog.fittedvalues
                df[f'{list(var.keys())[0]}_hat'] = X_hat
        df = sm.add_constant(df)        
        structural_model = sm.OLS(y,df).fit()
        modelos.update({'structural_model':structural_model})
        modelosPorZtree.update({ztree_iter : modelos})
        ztree_iter += 1
    return modelosPorZtree
            

        

In [79]:
y = mRNA.copy().reset_index()
y['Grupo'] = np.where(y['index'].str.contains('A'), 'Tratamiento', 
              np.where(y['index'].str.contains('C'), 'Control', 'Otro'))
y = y.set_index('index')
y = pd.get_dummies(y, columns=['Grupo'], drop_first=True)
y['Grupo_Tratamiento'] = y['Grupo_Tratamiento'].astype(int)
y = y['Grupo_Tratamiento']
X = scl(mRNA.copy())
Z = scl(lncRNA.copy())

In [84]:
zTree1 = instrumentos_validos(y = y,X = X,Z = Z)

TNNT1 causa Y a través de instrumentos válidos y exógenos: ['MBNL1-AS1', 'TNRC6C-AS1']
FGF12 causa Y a través de instrumentos válidos y exógenos: ['MIR1-1HG-AS1', 'LINC02208']


In [150]:
zTree1

[['MYL2', 'FGF12', {'TNNT1': ['MBNL1-AS1', 'TNRC6C-AS1']}],
 ['MYL2', 'LTBP2', 'SORT1', 'MFGE8', {'FGF12': ['MIR1-1HG-AS1', 'LINC02208']}]]

Ahora probamos a estudiar la causalidad pero no sólo usando los lncRNA que diferenciaban los grupos de control y tratamiento sino todos

In [88]:
df_noCoding = pd.read_excel('../mRNA_seq/mRNA_seq_AS_Control_LncRNA_identificados.xlsx')
df_noCoding = df_noCoding.loc[:, ~df_noCoding.columns.str.contains('Norm')]
#Eliminamos todas las variables que Norm ya que prefiero hacer nuestra propia normalizacion
df_noCoding = df_noCoding.set_index('hgnc_symbol').drop(['description','gene_biotype','chromosome_name'],axis = 1).T
Z2 = scl(df_noCoding.copy())
Z2 = Z2[['LINC00342', 'SSTR5-AS1', 'OLMALINC', 'LINC01405', 'NEAT1', 'LINC01936', 'IDI2-AS1', 'FTX', 'LINC00881', 'CARMN', 'LINC01278', 'GATA6-AS1', 'KCNQ1OT1', 'SNHG6', 'OTUD6B-AS1', 'MIR1-1HG-AS1', 'MALAT1', 'LINC00662', 'TRDN-AS1', 'TTTY10', 'H19', 'GABPB1-AS1', 'MIR29B2CHG', 'NNT-AS1', 'THAP9-AS1', 'LINC02762', 'STAG3L5P-PVRIG2P-PILRB', 'HIF1A-AS3', 'IRF1-AS1', 'LINC01608', 'PAX8-AS1', 'WAC-AS1', 'PLBD1-AS1', 'MIR99AHG', 'CTBP1-DT', 'LINC01184', 'GUSBP11', 'RTCA-AS1', 'SNHG14', 'LINC02503', 'LINC00667', 'BANCR', 'XIST', 'LINC01128', 'TTN-AS1', 'MIR22HG', 'NIPBL-DT', 'MEG3', 'MIR646HG', 'TMEM161B-AS1', 'SNHG29', 'HAND2-AS1', 'NRSN2-AS1', 'TUG1', 'DTX2P1-UPK3BP1-PMS2P11', 'HCG18', 'SH3BP5-AS1', 'UGDH-AS1', 'HCG11', 'LINC02208', 'LINC01719', 'MIR100HG', 'LINC02693', 'TPT1-AS1', 'VIM-AS1', 'FGD5-AS1', 'EBLN3P', 'LINC01578', 'LINC00963', 'THUMPD3-AS1', 'RASSF8-AS1', 'NAV2-AS2', 'HELLPAR', 'LINC01001', 'HCP5', 'ZNF667-AS1', 'JPX', 'MBNL1-AS1', 'LINC00472', 'LINC00504', 'TBX5-AS1', 'MAGI2-AS3', 'OIP5-AS1', 'NORAD', 'TSBP1-AS1', 'LINC02269', 'DANCR', 'PINK1-AS', 'SNHG32', 'SNHG16', 'NUTM2B-AS1', 'PRKG1-AS1', 'LINC00702', 'ERVK13-1', 'PSMA3-AS1', 'NUTM2A-AS1', 'ILF3-DT', 'CRNDE', 'TNRC6C-AS1', 'MUC20-OT1', 'PRDM16-DT', 'LINC-PINT', 'GARS-DT', 'ZNF710-AS1', 'MIR133A1HG', 'LINC00630']]

In [152]:
z_tree = instrumentos_validos_ZGrande(y = y,X = X,Z = Z2, q_interactions = 100)

In [165]:
z_tree[0]

[['MYL2', 'PACS1', 'FGF12', {'TNNT1': ['RASSF8-AS1', 'HCG11']}],
 ['CYP2J2', 'FGF12', {'MYL2': ['RASSF8-AS1', 'HCG11']}]]

In [166]:
estimacion_IV(y = y,X = X,Z = Z2,zTree = z_tree[0])

{0: {'instrumentModel': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282c066c890>,
  'structural_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282bdd45590>},
 1: {'instrumentModel': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282c0699bd0>,
  'structural_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282c06054d0>}}

In [170]:
modelos[tree]

{'instrumentModel': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282bddf5150>,
 'structural_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x282bddd4850>}

In [181]:
filas = []
for i in z_tree.values():
    modelos = estimacion_IV(y = y,X = X,Z = Z2,zTree = i)
    for tree in range(len(modelos)):
        model = modelos[tree]
        instrumentModel = model['instrumentModel']
        structural_model = model['structural_model']
        PM_R2 = structural_model.rsquared
        IM_R2 = instrumentModel.rsquared
        fila = {
            'PM_R2': PM_R2,
            'principal_model':structural_model,
            'IM_R2': IM_R2,
            'instrumentModel':instrumentModel,
            'tree': i[tree]
        }
        filas.append(fila)
df = pd.DataFrame(filas).sort_values('PM_R2',ascending=False).reset_index(drop=True)

In [189]:
df

,PM_R2,principal_model,IM_R2,instrumentModel,tree
0,0.790915,<statsmodels.regression.linear_model.Regressio...,0.188571,<statsmodels.regression.linear_model.Regressio...,"[MYL2, FGF12, {'HSPB6': ['LINC02693', 'DANCR']}]"
1,0.786371,<statsmodels.regression.linear_model.Regressio...,0.629004,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'TNNT1': ['TBX5-AS1', 'G..."
2,0.770577,<statsmodels.regression.linear_model.Regressio...,0.387827,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'HSPB6': ['TBX5-AS1', 'G..."
3,0.763925,<statsmodels.regression.linear_model.Regressio...,0.303655,<statsmodels.regression.linear_model.Regressio...,"[TNNT1, MYL2, FGF12, {'CYP2J2': ['EBLN3P', 'FG..."
4,0.758834,<statsmodels.regression.linear_model.Regressio...,0.575966,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'ACTA1': ['TBX5-AS1', 'G..."
...,...,...,...,...,...
88,0.445023,<statsmodels.regression.linear_model.Regressio...,0.532289,<statsmodels.regression.linear_model.Regressio...,"[CYP2J2, {'MYL2': ['PSMA3-AS1', 'IDI2-AS1']}]"
89,0.399665,<statsmodels.regression.linear_model.Regressio...,0.429862,<statsmodels.regression.linear_model.Regressio...,"[CYP2J2, {'MYL2': ['SNHG32', 'DTX2P1-UPK3BP1-P..."
90,0.398533,<statsmodels.regression.linear_model.Regressio...,0.437381,<statsmodels.regression.linear_model.Regressio...,"[CYP2J2, {'MYL2': ['GABPB1-AS1', 'TUG1', 'SNHG..."
91,0.251101,<statsmodels.regression.linear_model.Regressio...,0.542717,<statsmodels.regression.linear_model.Regressio...,"[{'FGF12': ['LINC01719', 'MIR100HG', 'FGD5-AS1..."


In [182]:
df_highR2 = df[df['PM_R2'] > 0.75]
df_highR2

,PM_R2,principal_model,IM_R2,instrumentModel,tree
0,0.790915,<statsmodels.regression.linear_model.Regressio...,0.188571,<statsmodels.regression.linear_model.Regressio...,"[MYL2, FGF12, {'HSPB6': ['LINC02693', 'DANCR']}]"
1,0.786371,<statsmodels.regression.linear_model.Regressio...,0.629004,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'TNNT1': ['TBX5-AS1', 'G..."
2,0.770577,<statsmodels.regression.linear_model.Regressio...,0.387827,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'HSPB6': ['TBX5-AS1', 'G..."
3,0.763925,<statsmodels.regression.linear_model.Regressio...,0.303655,<statsmodels.regression.linear_model.Regressio...,"[TNNT1, MYL2, FGF12, {'CYP2J2': ['EBLN3P', 'FG..."
4,0.758834,<statsmodels.regression.linear_model.Regressio...,0.575966,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'ACTA1': ['TBX5-AS1', 'G..."
5,0.755412,<statsmodels.regression.linear_model.Regressio...,0.737804,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'MFGE8': ['LINC02762', '..."
6,0.752549,<statsmodels.regression.linear_model.Regressio...,0.641219,<statsmodels.regression.linear_model.Regressio...,"[MYL2, LTBP2, FGF12, {'CYP2J2': ['TBX5-AS1', '..."


In [188]:
df_highR2.loc[5]['tree']

['MYL2', 'LTBP2', 'FGF12', {'MFGE8': ['LINC02762', 'TNRC6C-AS1']}]

In [186]:
print(df_highR2.loc[5]['principal_model'].summary())

                            OLS Regression Results                            
Dep. Variable:      Grupo_Tratamiento   R-squared:                       0.755
Model:                            OLS   Adj. R-squared:                  0.723
Method:                 Least Squares   F-statistic:                     23.16
Date:                Tue, 01 Jul 2025   Prob (F-statistic):           8.27e-09
Time:                        12:27:12   Log-Likelihood:                 1.8384
No. Observations:                  35   AIC:                             6.323
Df Residuals:                      30   BIC:                             14.10
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6857      0.042     16.359      0.0

In [187]:
print(df_highR2.loc[5]['instrumentModel'].summary())

                            OLS Regression Results                            
Dep. Variable:                  MFGE8   R-squared:                       0.738
Model:                            OLS   Adj. R-squared:                  0.721
Method:                 Least Squares   F-statistic:                     45.02
Date:                Tue, 01 Jul 2025   Prob (F-statistic):           4.99e-10
Time:                        12:27:17   Log-Likelihood:                -26.236
No. Observations:                  35   AIC:                             58.47
Df Residuals:                      32   BIC:                             63.14
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       8.327e-17      0.091    9.2e-16      1.0

LINC02762 / TNRC6C-AS1  →  ↑ MFGE8  →  ↓ Riesgo de enfermedad


# Borrador

In [331]:
from itertools import combinations_with_replacement

X_inter = pd.DataFrame()
for i, j in combinations_with_replacement(X.columns, 2):
    X_inter[f'{i}_{j}'] = X[i] * X[j]

In [337]:
from itertools import product

X_Z = pd.DataFrame()
for i, j in product(X.columns, Z.columns):
    X_Z[f'{i}_{j}'] = X[i] * Z[j]

In [ ]:
def interacciones_significativas (y,X,Z,q_interactions):
    from itertools import product
    import random
    from collections import defaultdict

    X_Z = pd.DataFrame()
    for i, j in product(X.columns, Z.columns):
        X_Z[f'{i}_{j}'] = X[i] * Z[j]
    #X_Z = X_Z.dropna(axis=1)
    columnas =list(X_Z.columns)
    conteo_significancia = defaultdict(int)
    for _ in range(q_interactions):
        seleccionadas = random.sample(columnas, min(10, len(columnas)))
        X_block = sm.add_constant(X_Z[seleccionadas])

        model = sm.OLS(y, X_block).fit()
        pvals = model.pvalues
        for var in pvals[pvals < 0.1].index:
                conteo_significancia[var] += 1
    df_resultado = pd.DataFrame.from_dict(conteo_significancia, orient='index', columns=['conteo_significancia'])
    df_resultado = df_resultado.sort_values(by='conteo_significancia', ascending=False)
    return df_resultado

In [406]:
interacciones_significativas (y,X,Z,5000)

,conteo_significancia
const,5000
MFGE8_TNRC6C-AS1,167
TNNT1_TNRC6C-AS1,163
MYL2_MBNL1-AS1,159
PACS1_LINC01278,153
...,...
XPR1_LINC02208,3
TNNT1_LINC00702,3
CYP2J2_LINC02208,3
HSPB6_LINC02208,2


In [ ]:
def interacciones_significativas (X,y):
    from itertools import combinations_with_replacement

    X_trans = pd.DataFrame()
    for i, j in combinations_with_replacement(X.columns, 2):
        X_trans[f'{i}_{j}'] = X[i] * X[j]
    lista_vars = list(range(0,len(X_trans.columns),10))
    lista_vars.append(len(X_trans.columns))
    significant_vars_list = []
    for i in range(len(lista_vars)-1):
        start = lista_vars[i]
        end = lista_vars[i + 1]
        X_const = sm.add_constant(X_trans[list(X_trans.columns[start:end])])
        model = sm.OLS(y, X_const).fit()
        model.summary()
        pvals = model.pvalues
        significant_vars_i = pvals[pvals < 0.1].index.tolist()
        significant_vars_list.append(significant_vars_i)
    significant_vars = list(set([j for i in significant_vars_list for j in i]))
    significant_vars.remove("const")
    return X_trans[significant_vars]